# Study 10 — Clustering Robustness

Auto-navigate to DB, analyze cluster state.

In [ ]:
import os, sqlite3, pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

# Auto-navigate to repo root
notebook_dir = Path.cwd()
while not (notebook_dir / 'data/db/pathosphere.db').exists():
    notebook_dir = notebook_dir.parent
    if notebook_dir == notebook_dir.parent: break
os.chdir(notebook_dir)

DB = Path('data/db/pathosphere.db').resolve()
print(f'CWD: {Path.cwd()}')
print(f'DB: {DB}')
CONN = sqlite3.connect(str(DB))
CONN.row_factory = sqlite3.Row

In [ ]:
# Cluster state
rss = CONN.execute('''
    SELECT e.id, e.title, COUNT(ed.document_id) as size
    FROM events e LEFT JOIN event_documents ed ON e.id = ed.event_id
    WHERE e.origin IN ("rss","comtrade") GROUP BY e.id ORDER BY size DESC
''').fetchall()

sizes = [r['size'] for r in rss]
singleton_pct = 100*sum(1 for s in sizes if s == 1)/len(rss) if rss else 0
print(f'RSS events: {len(rss)}')
print(f'Singleton: {singleton_pct:.1f}%')
print(f'Mean size: {np.mean(sizes):.2f}' if sizes else 'No data')

print('\nTop 5 titles:')
for i, r in enumerate(rss[:5], 1):
    is_garbage = r['title'].startswith('||') if r['title'] else False
    mark = '❌' if is_garbage else '✓'
    print(f'  {i}. [{r["size"]:2}] {mark} {r["title"][:70]}')

In [ ]:
# Garbage title check
garbage = CONN.execute("SELECT COUNT(*) as c FROM events WHERE title LIKE '||%'").fetchone()['c']
print(f'GDELT garbage titles (||...): {garbage}')
print(f'\nVERDICT: Clustering {"SOLID ✓" if garbage == 0 else "BROKEN ❌"}')